In [36]:
using Pkg
Pkg.activate("C:/Users/ibzja/Documents/UPF_2022_2026/4t/2n_trimestre/Practiques_tutelades/CellBasedModels.jl")
using CellBasedModels 
using GeometryBasics
using Distributions
using GLMakie
Makie.inline!(true)
using CSV, DataFrames, Statistics
using Printf, JLD2


  Activating project at `C:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl`


In [37]:
rods2DChemotaxis = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  #Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :fx => Float64,
        :fy => Float64,
        :W => Float64,
        :pressure => Float64,
        :active => Bool,

        :isSource => Bool, 
        :S => Float64,

        :methyl => Float64, #Receptor methylation
        :Yp => Float64, #CheYP levels, probability of tumblingç
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64,
        :M => Float64

        # :xs => Float64,
        # :ys => Float64
        
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, #Energy parameters
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, #Cooperativity
        :Ki => Float64, #Dissociation constants
        :Ka => Float64,
        :τm => Float64, #Methylation adaptation timescale
        :α => Float64,      #Total Yp pool
        :ωFrec => Float64,     #Basal switching frequency
        :Ky => Float64,         #CheA - CheY phosphorylation rate
        :Z => Float64,          #CheZ concentration
        :Kz => Float64,         #CheZ mediated dephosphorylation rate
        :Yy => Float64,         #Basa Yp leak

        :DMedium => Float64,
        :DSource => Float64,
        :delta => Float64 
    ),

    medium = Dict(
        :mm => Float64
    ),

    agentODE = quote

        M = mm


        F = ε0 + ε1 * methyl + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) #Equació del paper per definir activitat del receptor
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)        #Energia lliure en estat adaptat

        mx = (ε0 + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) - F0) / (- ε1)

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))      #CheYp segons activitat receptor

        dt(x) = active * vx  #Change position acording to constant speed afected by forces
        dt(y) = active * vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)        #Methylation
        
    end,

    agentRule = quote
        if isSource
            active = false
            vx = v * cos(theta)
            vy = v * sin(theta)
            x += vx * dt
            y += vy * dt
            theta += sqrt(2 * DSource * dt) * randn()
            mm += S  
        else

            v_run = v
            v_tumble = 0.25 

            speed = active ? v_run : v_tumble

            Dr_tumble = 6.2      
            Dr_total = active ? Dr_run : Dr_tumble

            if active 
                λ = ωFrec*exp(-G)
                P = 1 - exp(-λ * dt)
                
            else
                λ = ωFrec*exp(G)
                P = 1 - exp(-λ * dt)  
                
            end


            if active 
                λrt = ωFrec*exp(-G) 

                P_rt = 1 - exp(-λrt * dt)
                P = rand() 
                                                    #Si rate alta = mes probabilitat de canvi. Per tant, si random number mes petit =  canvi. 
                if P < P_rt             #Si rate alta = mes probabilitat de canvi. Per tant, si random number mes petit =  canvi. 
                    active = false
                    vx = speed
                    vy = speed
                    theta += sqrt(2 * Dr_total * dt) * randn()           #Tumble = random reorientation
                else     #Si rate baixa 
                    active = true
                    vx = speed * cos(theta)
                    vy = speed * sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()       #Keep running, reorientation according to rotational difusion
                end

            elseif !active
                λtr = ωFrec*exp(G) 
                P_tr = 1 - exp(-λtr * dt)
                P = rand()

                if P < P_tr
                    active = true
                    vx = speed * cos(theta)
                    vy = speed * sin(theta)
                    theta += sqrt(2 * Dr_total * dt) * randn()
                else
                    active = false
                    vx = speed
                    vy = speed
                    theta += sqrt(2 * Dr_total * dt) * randn()
                end
            end
        end

        collision = (x ≤ simBox[1,1]) || (x ≥ simBox[1,2]) || (y ≤ simBox[2,1]) || (y ≥ simBox[2,2])

        if collision
            vx = 0.25
            vy = 0.25
            theta = rand(Uniform(0, 2π))
        end


    end,

    mediumODE = quote
        if @mediumInside()
            dt(mm) = DMedium *(@∂2(1, mm)+ @∂2(2, mm)) - delta*mm 
        elseif @mediumBorder(1,-1)
            mm = 0
        elseif @mediumBorder(1,1)
            mm = 0
        elseif @mediumBorder(2,-1)
            mm = 0
        elseif @mediumBorder(2,-1)
            mm = 0
        end
    end,

    neighborsAlg = CBMNeighbors.CellLinked(cellEdge = 4)
)

PARAMETERS
	x (Float64 agent)
	y (Float64 agent)
	xₘ (Float64 medium)
	yₘ (Float64 medium)
	F (Float64 agent)
	active (Bool agent)
	methyl (Float64 agent)
	l (Float64 agent)
	S (Float64 agent)
	M (Float64 agent)
	d (Float64 agent)
	λ (Float64 agent)
	v (Float64 agent)
	isSource (Bool agent)
	A (Float64 agent)
	fx (Float64 agent)
	vx (Float64 agent)
	fy (Float64 agent)
	m (Float64 agent)
	Yp (Float64 agent)
	P (Float64 agent)
	pressure (Float64 agent)
	vy (Float64 agent)
	W (Float64 agent)
	G (Float64 agent)
	theta (Float64 agent)
	ε1 (Float64 model)
	α (Float64 model)
	Z (Float64 model)
	DSource (Float64 model)
	Dr_run (Float64 model)
	Ka (Float64 model)
	ε3 (Float64 model)
	ε0 (Float64 model)
	DMedium (Float64 model)
	delta (Float64 model)
	Ky (Float64 model)
	Kz (Float64 model)
	K (Float64 model)
	ε2 (Float64 model)
	Nrec (Float64 model)
	τm (Float64 model)
	Yy (Float64 model)
	Ki (Float64 model)
	ωFrec (Float64 model)
	mm (Float64 medium)


UPDATE RULES
mediumODE
 if @mediumInside()

In [38]:
com = Community(
    rods2DChemotaxis,
    N=1,
    dt=0.01,
    # simBox = [-20.0 20.0; -20.0 20.0],
    simBox = [-50.0 50.0; -50.0 50.0],
    NMedium = [100, 100]
    # NMedium = [200, 200]
)

m = 1/100
g = 1/10000
d = 1

com.Dr_run = 0.062

com.v =  0.33   #Velocitat neutrophil (20 microm/min) 
# com.v = 20.0    #(microm/s)
# com.v = 10.0
# com.v = 5.0
# com.v = 2.5
# com.v = 1

# com.DMedium = 100
# com.DMedium = 10
com.DMedium = 1
# com.DMedium = 0.1
# com.DMedium = 0.01
# com.DMedium = 0.001
# com.DMedium = 0.0001

com.DSource = 0.063
com.delta = 0.01

com.ωFrec = 1.3
com.Ki = 0.0182
com.Ka = 3.0
com.Nrec = 6.0
com.ε0   = 6.0
com.ε1   = -1.0
com.ε2   = 80
com.ε3   = 80

# com.τm = 5.0
# com.τm = 10.0
com.τm = 30.0

com.α   = 6.0

com.K = 2.0 

com.Ky = 100.0
com.Kz = 10.0
com.Z = 5.0
com.Yy = 0.1

com.m = 1.        
com.d = 1.        
com.l = 3;

com.x = rand(Uniform(com.simBox[1,:]...),com.N)
com.y = rand(Uniform(com.simBox[2,:]...),com.N)
com.theta = rand(Uniform(0,2π),com.N)

com.methyl .= 0.0
com.Yp .= com.K

src = 1

com.isSource .= false
com.isSource[src] = true

com.active .= true 
com.active[src] = false   

com.S .= 0.0
# com.xs .= 0.0
# com.ys .= 0.0
# com.xs[src] = com.x[src]
# com.ys[src] = com.y[src]

# com.S[src] = 0.0025
# com.S[src] = 0.016
com.S[src] = 0.1
# com.S[src] = 100
# com.S[src] = 1000


0.1

In [39]:
outfile = "prey_test.jld2"
steps = 10000

loadToPlatform!(com, preallocateAgents=1)
com.mm = zeros(Float64, com.NMedium...)

jldopen(outfile, "w") do file

    # -------------------------
    # Metadata (written once)
    # -------------------------
    meta = JLD2.Group(file, "meta")
    meta["N"] = com.N
    meta["NMedium"] = com.NMedium
    meta["steps"] = steps

    # -------------------------
    # Main loop
    # -------------------------
    for step in 1:steps
        step!(com)

        stepname = @sprintf("step_%06d", step)
        g = JLD2.Group(file, stepname)

        # Agent-level arrays (length = N)
        g["x"] = copy(com.x)
        g["y"] = copy(com.y)
        g["theta"] = copy(com.theta)
        g["l"] = copy(com.l)
        g["d"] = copy(com.d)
        g["active"] = copy(com.active)
        g["isSource"] = copy(com.isSource)

        # Medium grid (saved once per step)
        g["mm_grid"] = copy(com.mm)
    end
end


# Once difusion loaded for that agent, take that grid matrix for other agents. 


In [40]:
function grid_value(mm, x, y, dx, dy)
    i = clamp(Int(floor(x / dx)) + 1, 1, size(mm,1))
    j = clamp(Int(floor(y / dy)) + 1, 1, size(mm,2))
    return mm[i, j]
end

grid_value (generic function with 1 method)

In [56]:
rods2DChemotaxis_test = ABM(2,
    agent = Dict(
        :vx => Float64,
        :vy => Float64,
        :v => Float64,  #Swimming speed
        :theta => Float64,
        :d => Float64,
        :l => Float64,
        :m => Float64,
        :fx => Float64,
        :fy => Float64,
        :W => Float64,
        :pressure => Float64,
        :active => Bool,

        :methyl => Float64, #Receptor methylation
        :Yp => Float64, #CheYP levels, probability of tumblingç
        :G => Float64,
        :λ => Float64,
        :P => Float64,
        :M => Float64,
        :F => Float64,
        :A => Float64,
        :M => Float64

        
    ),

    model = Dict(

        :Dr_run => Float64,

        :ε0 => Float64, #Energy parameters
        :ε1 => Float64,
        :ε2 => Float64,
        :ε3 => Float64,
        :K => Float64,
        :Nrec => Float64, #Cooperativity
        :Ki => Float64, #Dissociation constants
        :Ka => Float64,
        :τm => Float64, #Methylation adaptation timescale
        :α => Float64,      #Total Yp pool
        :ωFrec => Float64,     #Basal switching frequency
        :Ky => Float64,         #CheA - CheY phosphorylation rate
        :Z => Float64,          #CheZ concentration
        :Kz => Float64,         #CheZ mediated dephosphorylation rate
        :Yy => Float64           #Basa Yp leak

    ),

    agentODE = quote
        time = step
        if isinf(x) || isinf(y) || isnan(x) || isnan(y)
            @warn "Agent position is invalid!" x y
            # Optionally: skip or reset agent
            continue
        end
        
        val = jldopen("prey_test.jld2", "r") do file
            file[@sprintf("step_%06d/mm_grid", time)]
        end
        
        mm = grid_value(val, x, y, dx, dy)

        M = mm


        F = ε0 + ε1 * methyl + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) #Equació del paper per definir activitat del receptor
        F0 = log(((Ky * (α - K)) / (K * (Kz * Z + Yy))) - 1)        #Energia lliure en estat adaptat

        mx = (ε0 + Nrec * log((1 + mm / Ki) / (1 + mm / Ka)) - F0) / (- ε1)

        A = 1 / (1 + exp(F))    

        Yp = (Ky * A * α) / ((Ky * A) + (Kz * Z) + Yy)

        G = ε2 / 4 - (ε3 / 2) / (1 + (K / Yp))      #CheYp segons activitat receptor

        dt(x) = active * vx  #Change position acording to constant speed afected by forces
        dt(y) = active * vy  
        dt(methyl) = -(1 / τm) * (methyl - mx)        #Methylation
        
    end,

    agentRule = quote
     
        v_run = v
        v_tumble = 0.25 

        speed = active ? v_run : v_tumble

        Dr_tumble = 6.2      
        Dr_total = active ? Dr_run : Dr_tumble

        if active 
            λ = ωFrec*exp(-G)
            P = 1 - exp(-λ * dt)
                
        else
            λ = ωFrec*exp(G)
            P = 1 - exp(-λ * dt)  
                
        end


        if active 
            λrt = ωFrec*exp(-G) 

            P_rt = 1 - exp(-λrt * dt)
            P = rand() 
                                                    #Si rate alta = mes probabilitat de canvi. Per tant, si random number mes petit =  canvi. 
            if P < P_rt             #Si rate alta = mes probabilitat de canvi. Per tant, si random number mes petit =  canvi. 
                active = false
                vx = speed
                vy = speed
                theta += sqrt(2 * Dr_total * dt) * randn()           #Tumble = random reorientation
            else     #Si rate baixa 
                active = true
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()       #Keep running, reorientation according to rotational difusion
            end

        elseif !active
            λtr = ωFrec*exp(G) 
            P_tr = 1 - exp(-λtr * dt)
            P = rand()

            if P < P_tr
                active = true
                vx = speed * cos(theta)
                vy = speed * sin(theta)
                theta += sqrt(2 * Dr_total * dt) * randn()
            else
                active = false
                vx = speed
                vy = speed
                theta += sqrt(2 * Dr_total * dt) * randn()
            end
        end

        collision = (x ≤ simBox[1,1]) || (x ≥ simBox[1,2]) || (y ≤ simBox[2,1]) || (y ≥ simBox[2,2])

        if collision
            vx = v_tumble
            vy = v_tumble
            theta = rand(Uniform(0, 2π))
        end


    end,

    # mediumODE = quote
    #     if @mediumInside()
    #         dt(mm) = DMedium *(@∂2(1, mm)+ @∂2(2, mm)) - delta*mm 
    #     elseif @mediumBorder(1,-1)
    #         mm = 0
    #     elseif @mediumBorder(1,1)
    #         mm = 0
    #     elseif @mediumBorder(2,-1)
    #         mm = 0
    #     elseif @mediumBorder(2,-1)
    #         mm = 0
    #     end
    # end,

    neighborsAlg = CBMNeighbors.CellLinked(cellEdge = 4)
)

PARAMETERS
	x (Float64 agent)
	y (Float64 agent)
	M (Float64 agent)
	d (Float64 agent)
	active (Bool agent)
	v (Float64 agent)
	λ (Float64 agent)
	F (Float64 agent)
	fx (Float64 agent)
	A (Float64 agent)
	methyl (Float64 agent)
	vx (Float64 agent)
	l (Float64 agent)
	m (Float64 agent)
	fy (Float64 agent)
	Yp (Float64 agent)
	P (Float64 agent)
	pressure (Float64 agent)
	vy (Float64 agent)
	W (Float64 agent)
	G (Float64 agent)
	theta (Float64 agent)
	ε1 (Float64 model)
	α (Float64 model)
	Z (Float64 model)
	Dr_run (Float64 model)
	Ka (Float64 model)
	ε3 (Float64 model)
	ε0 (Float64 model)
	Ky (Float64 model)
	Kz (Float64 model)
	K (Float64 model)
	ε2 (Float64 model)
	Nrec (Float64 model)
	τm (Float64 model)
	Yy (Float64 model)
	Ki (Float64 model)
	ωFrec (Float64 model)


UPDATE RULES
agentRule
 begin
    v_run = v
    v_tumble = 0.25
    speed = if active
            v_run
        else
            v_tumble
        end
    Dr_tumble = 6.2
    Dr_total = if active
            Dr_run
      

In [59]:
com_test = Community(
    rods2DChemotaxis_test,
    N=20,
    dt=0.01,
    # simBox = [-20.0 20.0; -20.0 20.0],
    simBox = [0 100; 0 100],
    NMedium = [100, 100]
    # NMedium = [200, 200]
)

m = 1/100
g = 1/10000
d = 1

com_test.Dr_run = 0.062

# com_test.v =  0.33   #Velocitat neutrophil (20 microm/min) 
com_test.v = 20.0    #(microm/s)
# com_test.v = 10.0
# com_test.v = 5.0
# com_test.v = 2.5
# com_test.v = 1

com_test.ωFrec = 1.3
com_test.Ki = 0.0182
com_test.Ka = 3.0
com_test.Nrec = 6.0
com_test.ε0   = 6.0
com_test.ε1   = -1.0
com_test.ε2   = 80
com_test.ε3   = 80

# com_test.τm = 5.0
# com_test.τm = 10.0
com_test.τm = 30.0

com_test.α   = 6.0

com_test.K = 2.0 

com_test.Ky = 100.0
com_test.Kz = 10.0
com_test.Z = 5.0
com_test.Yy = 0.1

com_test.m = 1.        
com_test.d = 1.        
com_test.l = 3;

com_test.x = rand(Uniform(com_test.simBox[1,:]...),com_test.N)
com_test.y = rand(Uniform(com_test.simBox[2,:]...),com_test.N)
com_test.theta = rand(Uniform(0,2π),com_test.N)

com_test.methyl .= 0.0
com_test.Yp .= com_test.K

com_test.active .= true 

20-element Vector{Bool}:
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1
 1

In [60]:
outfile = "prey_test_agents.jld2"
steps = 10000

loadToPlatform!(com_test, preallocateAgents=20)

jldopen(outfile, "w") do file

    # -------------------------
    # Metadata (written once)
    # -------------------------
    meta = JLD2.Group(file, "meta")
    meta["N"] = com_test.N
    meta["NMedium"] = com_test.NMedium
    meta["steps"] = steps

    # -------------------------
    # Main loop
    # -------------------------
    for step in 1:steps
        step!(com_test)

        stepname = @sprintf("step_%06d", step)
        g = JLD2.Group(file, stepname)

        # Agent-level arrays (length = N)
        g["x"] = copy(com_test.x)
        g["y"] = copy(com_test.y)
        g["theta"] = copy(com_test.theta)
        g["l"] = copy(com_test.l)
        g["d"] = copy(com_test.d)
        g["active"] = copy(com_test.active)

        # Medium grid (saved once per step)
        g["mm"] = copy(com_test.M)
    end
end


CompositeException: TaskFailedException

    nested task error: InexactError: Int64(Inf)
    Stacktrace:
     [1] Int64
       @ .\float.jl:912 [inlined]
     [2] grid_value(mm::Matrix{Float64}, x::Float64, y::Float64, dx::Float64, dy::Float64)
       @ Main c:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl\examples\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X31sZmlsZQ==.jl:2
     [3] macro expansion
       @ c:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl\examples\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X30sZmlsZQ==.jl:63 [inlined]
     [4] macro expansion
       @ C:\Users\ibzja\Documents\UPF_2022_2026\4t\2n_trimestre\Practiques_tutelades\CellBasedModels.jl\src\AgentStructure\auxiliar.jl:166 [inlined]
     [5] (::var"#327#threadsfor_fun#225"{var"#327#threadsfor_fun#222#226"{SubArray{Float64, 1, Matrix{Float64}, Tuple{Int64, Base.Slice{Base.OneTo{Int64}}}, true}, SubArray{Float